# NTX Q1 Stage 22 — Safety–Utility–Latency Frontier

Combines completed guarded/baseline result files to show whether safety gains come from meaningful authorization rather than indiscriminate blocking or clarification.

**Evidence rule:** failed/incomplete third-party infrastructure is recorded as such; no missing score is converted into a model failure or a success.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time, hashlib, zipfile, random
BUNDLE = Path(os.environ.get("NTX_NOTEBOOK_BUNDLE", ".")).resolve()
if str(BUNDLE) not in sys.path:
    sys.path.insert(0, str(BUNDLE))
from ntx_q1_shared import *
ROOT = ensure_dir(DEFAULT_ROOT)
RESULTS_ROOT = ensure_dir(DEFAULT_RESULTS)
SEED = int(os.environ.get("NTX_SEED", "42"))
random.seed(SEED)
print("ROOT:", ROOT)
print("RESULTS:", RESULTS_ROOT)
print("Frozen repo commit:", REPO_COMMIT)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
STAGE="22_frontier"
OUT=ensure_dir(RESULTS_ROOT/STAGE)

# Input is a standardized method-level table. If absent, construct from known stage files when possible.
standard=ROOT/"inputs"/STAGE/"method_results.csv"
rows=[]
if standard.exists():
    methods=pd.read_csv(standard)
else:
    # pull Stage 19 baseline metrics
    p=RESULTS_ROOT/"19_baselines"/"baseline_metrics.csv"
    if p.exists():
        b=pd.read_csv(p)
        for _,r in b[b.status.eq("OK")].iterrows():
            rows.append({
                "method":r["variant"],
                "unsafe_allow_rate":r.get("unsafe_allow_rate",np.nan),
                "safe_action_completion":r.get("safe_action_completion",np.nan),
                "clarification_rate":r.get("clarification_rate",np.nan),
                "latency_median_ms":np.nan
            })
    # pull AgentDojo paired summary if present
    p=RESULTS_ROOT/"18_agentdojo_guarded"/"agentdojo_guarded_summary.json"
    if p.exists():
        s=json.loads(p.read_text())
        for m in ["unguarded","guarded"]:
            rows.append({"method":"agentdojo_"+m,**s[m],
                         "latency_median_ms":s.get("guard_latency_median_ms",np.nan) if m=="guarded" else 0})
    methods=pd.DataFrame(rows)

methods.to_csv(OUT/"frontier_input_normalized.csv",index=False)
display(methods)

In [ ]:
if len(methods):
    # Dominance: lower unsafe_allow_rate, higher safe_action_completion.
    pts=methods.dropna(subset=["unsafe_allow_rate","safe_action_completion"]).copy()
    pts["pareto"]=True
    for i,a in pts.iterrows():
        dominated=((pts.unsafe_allow_rate<=a.unsafe_allow_rate)&
                   (pts.safe_action_completion>=a.safe_action_completion)&
                   ((pts.unsafe_allow_rate<a.unsafe_allow_rate)|(pts.safe_action_completion>a.safe_action_completion))).any()
        pts.loc[i,"pareto"]=not dominated
    pts.to_csv(OUT/"pareto_frontier.csv",index=False)

    fig,ax=plt.subplots(figsize=(7,5))
    ax.scatter(pts.unsafe_allow_rate,pts.safe_action_completion)
    for _,r in pts.iterrows():
        ax.annotate(str(r.method),(r.unsafe_allow_rate,r.safe_action_completion),xytext=(4,4),textcoords="offset points",fontsize=8)
    ax.set_xlabel("Unsafe Allow Rate (lower is better)")
    ax.set_ylabel("Safe Action Completion (higher is better)")
    ax.set_title("NiyamTrace-X Safety–Utility Frontier")
    ax.grid(alpha=.25)
    fig.tight_layout()
    fig.savefig(OUT/"safety_utility_frontier.png",dpi=300)
    plt.show()

    record_status(OUT,STAGE,"PASS" if len(pts)>=3 else "PARTIAL",methods=int(len(pts)))
else:
    record_status(OUT,STAGE,"WAITING_FOR_METHOD_RESULTS")

In [ ]:
z=package_stage(OUT,"NTX_Q1_22_FRONTIER_RESULTS.zip")
print("Created:",z)
maybe_colab_download(z)